TK

<!-- <span class='.preview-image' style="display: none;"><img src='preview.png'></img></span> -->

<!-- **Run this notebook in the browser using Binder here [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/diyclassics/epblog/notebooks?labpath=notebooks%2Fmethod-counter.ipynb)** -->

MORE

In [1]:
# Preliminary imports

from collections import Counter
from pprint import pprint

In [2]:
# Downloading RNNTagger with tqdm progress bar
# ~15 minutes!

import urllib.request
from pathlib import Path
from tqdm import tqdm

class TqdmUpTo(tqdm):
    def update_to(self, block_num=1, block_size=1, total_size=None):
        if total_size is not None:
            self.total = total_size
        self.update(block_num * block_size - self.n)


tools_dir = Path('tools')
tools_dir.mkdir(exist_ok=True)
zip_path = tools_dir / 'RNNTagger-1.5.0.zip'
extracted_dir = tools_dir / 'RNNTagger'
url = 'https://www.cis.uni-muenchen.de/~schmid/tools/RNNTagger/data/RNNTagger-1.5.0.zip'

if zip_path.exists() or extracted_dir.exists():
    print(f'Skipping download: {zip_path if zip_path.exists() else extracted_dir} already exists.')
else:
    print(f'Downloading to {zip_path}...')
    with TqdmUpTo(unit='B', unit_scale=True, miniters=1, desc=zip_path.name) as t:
        urllib.request.urlretrieve(url, zip_path, reporthook=t.update_to)
    print('Download complete.')

Skipping download: tools/RNNTagger already exists.


In [3]:
# Make RNNTagger available
# ~6 minutes!

import zipfile
from pathlib import Path

if extracted_dir.exists():
    print(f'Skipping extraction: {extracted_dir} already exists.')
else:
    zip_path = Path('tools/RNNTagger-1.5.0.zip')
    extract_dir = Path('tools')

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print('Extraction complete.')

    zip_path.unlink()
    print('ZIP file deleted to save disk space.')


Skipping extraction: tools/RNNTagger already exists.


In [4]:
# Create test.txt and run RNNTagger shell script
import subprocess
import os
from pathlib import Path
from tabulate import tabulate

rnn_dir = Path('tools/RNNTagger')
cmd_script = 'cmd/rnn-tagger-latin.sh'  # Use relative path from rnn_dir
test_text= """Turpissima tamen est iactura quae per neglegentiam fit."""
test_file = 'test.txt'  # Use relative path from rnn_dir

# Ensure test.txt exists
test_file_path = rnn_dir / test_file
if not test_file_path.exists():
    with open(test_file_path, 'w') as f:
        f.write(test_text)

if not (rnn_dir / cmd_script).exists():
    raise FileNotFoundError(f'Shell script not found: {rnn_dir / cmd_script}')

# Ensure shell script is executable
if not os.access(rnn_dir / cmd_script, os.X_OK):
    os.chmod(rnn_dir / cmd_script, 0o755)

# Make all .pl and .perl scripts in scripts/ executable
scripts_dir = rnn_dir / 'scripts'
if scripts_dir.exists():
    for script in scripts_dir.iterdir():
        if script.suffix in ['.pl', '.perl'] and script.is_file():
            if not os.access(script, os.X_OK):
                os.chmod(script, 0o755)

# Also check posts/tool-tools location
alt_scripts_dir = Path('posts/tool-tools/RNNTagger/scripts')
if alt_scripts_dir.exists():
    for script in alt_scripts_dir.iterdir():
        if script.suffix in ['.pl', '.perl'] and script.is_file():
            if not os.access(script, os.X_OK):
                os.chmod(script, 0o755)

# Run the shell script and capture output
try:
    result = subprocess.run(['bash', cmd_script, test_file], cwd=rnn_dir, capture_output=True, text=True, check=True)
    output_lines = [line for line in result.stdout.strip().split('\n') if line]
    
    # Try to split each line into columns (tab or space delimited)
    table = [line.split() for line in output_lines]
    print(tabulate(table, headers=["word", "tag", "lemma"]))
    # if result.stderr:
    #     print(result.stderr)
except subprocess.CalledProcessError as e:
    print('Error running shell script:')
    print('Return code:', e.returncode)
    print('Output:', e.output)
    print('STDERR:', e.stderr)

word          tag          lemma
------------  -----------  -----------
Turpissima    13A---A2---  turpus
tamen         4-O--------  tamen
est           3-NA1--6---  sum
iactura       11A---A2---  iactura
quae          11F---A2--1  qui
per           4-S--------  per
neglegentiam  11A---D2---  neglegentia
fit           3-NJ1--6---  facio
.             SENT         .


In [5]:
import pandas as pd

# Load the tagset TSV
tagset_df = pd.read_csv('latin-tagset.tsv', sep='\t')

def get_tag_category(tag, pos):
    """
    Given a tag string and position (1-based), return the human-readable meaning for that character.
    """
    if pos > len(tag):
        return "-"
    value = tag[pos-1]
    # Find the row matching position and code
    row = tagset_df[(tagset_df['ID'] == pos) & (tagset_df['Code'] == value)]
    if not row.empty:
        return row.iloc[0]['Value']
    else:
        return "-"
    
max_position = 11
category_names = [
    "Flexional-Type",
    "Nominals-Degree",
    "Flexional-Category",
    "Mood",
    "Tense",
    "Participials-Degree",
    "Case/Number",
    "Gender/Number/Person",
    "Composition",
    "Formal-Variation",
    "Graphical-Variation"
]

rows = []
for word, tag, lemma in table:
    row_dict = {'word': word, 'tag': tag, 'lemma': lemma}
    for pos in range(1, max_position + 1):
        col_name = f"{category_names[pos-1]}_category" if pos <= len(category_names) else f"pos{pos}_category"
        row_dict[col_name] = get_tag_category(tag, pos)
    rows.append(row_dict)

df_multi = pd.DataFrame(rows)
df_multi

,word,tag,lemma,Flexional-Type_category,Nominals-Degree_category,Flexional-Category_category,Mood_category,Tense_category,Participials-Degree_category,Case/Number_category,Gender/Number/Person_category,Composition_category,Formal-Variation_category,Graphical-Variation_category
0,Turpissima,13A---A2---,turpus,Nominal (only degrees and cases),Superlative,I decl,NaN,NaN,NaN,Singular Nominative,Feminine,NaN,NaN,NaN
1,tamen,4-O--------,tamen,Invariable,NaN,Invariable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,est,3-NA1--6---,sum,Verbal,NaN,Regularly irregular conjug,Active indicative,Present,NaN,NaN,III singular,NaN,NaN,NaN
3,iactura,11A---A2---,iactura,Nominal (only degrees and cases),Positive,I decl,NaN,NaN,NaN,Singular Nominative,Feminine,NaN,NaN,NaN
4,quae,11F---A2--1,qui,Nominal (only degrees and cases),Positive,Regularly irregular decl,NaN,NaN,NaN,Singular Nominative,Feminine,NaN,NaN,Baseform
5,per,4-S--------,per,Invariable,NaN,Prepositional (always or not) particle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,neglegentiam,11A---D2---,neglegentia,Nominal (only degrees and cases),Positive,I decl,NaN,NaN,NaN,Singular Accusative,Feminine,NaN,NaN,NaN
7,fit,3-NJ1--6---,facio,Verbal,NaN,Regularly irregular conjug,Pass/Dep indicative,Present,NaN,NaN,III singular,NaN,NaN,NaN
8,.,SENT,.,-,-,Regularly irregular conjug,-,-,-,-,-,-,-,-


### References

::: {#refs}
:::